# 04. Модель прогноза и ошибка

Ноутбук обучает модель прогноза `net_sales_qty`, использует последние 28 дней как holdout, сравнивает результат с лучшим baseline и сохраняет артефакты в `results/`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.config import PROCESSED_DATA_DIR, RESULTS_DIR
from src.features import build_feature_matrix
from src.modeling import (
    compare_with_best_baseline,
    evaluate_model,
    prepare_train_test,
    predict_non_negative,
    save_feature_importance,
    save_model_metrics,
    save_predictions,
    train_model,
)

## Загрузка признаков

In [ ]:
features_path = PROCESSED_DATA_DIR / 'features_lags_rolling.csv'
if features_path.exists():
    features = pd.read_csv(features_path, parse_dates=['sales_date'])
else:
    mart_daily_sales = pd.read_csv(PROCESSED_DATA_DIR / 'mart_daily_sales.csv', parse_dates=['sales_date'])
    features = build_feature_matrix(mart_daily_sales)
    features.to_csv(features_path, index=False)

features.head()

## Разделение на train и holdout

Holdout — последние 28 календарных дней. Такой порядок нужен, чтобы проверка имитировала прогноз будущего периода.

In [ ]:
train, holdout, feature_columns = prepare_train_test(features, holdout_days=28)
train[['sales_date', 'stock_code', 'market_id', 'net_sales_qty']].tail(), holdout[['sales_date', 'stock_code', 'market_id', 'net_sales_qty']].head()

## Обучение и прогноз

In [ ]:
model = train_model(train, feature_columns)
forecast = predict_non_negative(model, holdout, feature_columns)
model_metrics = evaluate_model(holdout['net_sales_qty'], forecast)
model_metrics

## Сравнение с baseline

In [ ]:
baseline_path = RESULTS_DIR / 'baseline_metrics.csv'
baseline_metrics = pd.read_csv(baseline_path) if baseline_path.exists() else None
comparison = compare_with_best_baseline(model_metrics, baseline_metrics)
comparison

## Сохранение результатов

In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
save_model_metrics(model_metrics, RESULTS_DIR / 'model_metrics.csv')
save_predictions(holdout, forecast, RESULTS_DIR / 'forecast_vs_actual_sample.csv')
feature_importance_path = save_feature_importance(model, feature_columns, RESULTS_DIR / 'feature_importance.csv')
comparison.to_csv(RESULTS_DIR / 'model_vs_baseline.csv', index=False)
feature_importance_path

## Важность признаков

In [ ]:
if feature_importance_path is not None:
    feature_importance = pd.read_csv(feature_importance_path)
    display(feature_importance.head(20))
else:
    feature_importance = pd.DataFrame(columns=['feature', 'importance'])
    print('Текущая модель не предоставляет важность признаков.')

## График факт против прогноза

In [ ]:
plot_frame = holdout[['sales_date', 'stock_code', 'market_id', 'net_sales_qty']].copy()
plot_frame['forecast_net_sales_qty'] = forecast.to_numpy()

sample_key = plot_frame.groupby(['stock_code', 'market_id'])['net_sales_qty'].sum().sort_values(ascending=False).index[0]
sample = plot_frame[(plot_frame['stock_code'] == sample_key[0]) & (plot_frame['market_id'] == sample_key[1])]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(sample['sales_date'], sample['net_sales_qty'], marker='o', label='Факт')
ax.plot(sample['sales_date'], sample['forecast_net_sales_qty'], marker='o', label='Прогноз')
ax.set_title(f'Факт и прогноз: {sample_key[0]} / {sample_key[1]}')
ax.set_xlabel('Дата')
ax.set_ylabel('net_sales_qty')
ax.legend()
plt.tight_layout()

## Выводы

- WMAPE модели: `[A]`.
- MAE модели: `[B]`.
- RMSE модели: `[C]`.
- Forecast bias: `[D]`.
- Сравнение с лучшим baseline по WMAPE: `[E]`.
- Если bias положительный, прогноз завышает спрос; если отрицательный, занижает.